# Introduction

You can launch a Jupyter notebook directly in the desired woring directory. Open anaconda prompt and move to your working directory:
```
cd 'D:/directory/of/your/choice'
```
then
```
jupyter notebook
```

# 1: EEG data preprocessing

## Import the relevant packages

In [1]:
import os
import mne
import pyprep
import csv
import numpy as np
from matplotlib import pyplot as plt
from pathlib import Path
# %matplotlib qt

## Data reading

In [ ]:
dir_path = Path(os.getcwd(),"Data","sub-001_task-CBOFF_run-1_eeg.set")

In [ ]:
raw = mne.io.read_raw_eeglab(dir_path, preload=True)

In [ ]:
raw.plot(
    duration=10,
    n_channels=20,
    decim=4,
    events=events,
    event_id=event_dict,
    event_color=None,
    theme='dark'
     )

## Assign the correct channel types

In [ ]:
print(raw.info)

In [ ]:
print(raw.info.ch_names)

In [ ]:
raw.get_channel_types()

In [ ]:
ch_type = {'ECG': 'ecg'}

In [ ]:
raw.set_channel_types(ch_type)

In [ ]:
print(raw.info)

In [ ]:
raw.get_channel_types()

In [ ]:
# Check number of channels and sampling rate

## Filtering

In [ ]:
hpf = 0.2  # (Hz) # to remove drifts, before epochs for edge artifacts
lpf = 100  # (Hz)
raw_filt = raw.copy().filter(hpf, lpf)

In [ ]:
notch = None  # (Hz)
raw_filt.notch_filter(notch) if notch is not None else None

## Referencing

In [ ]:
# ref = ['left_ear', 'right_ear']
raw.set_eeg_reference("average")
# Usually average AFTER the bad channels
# # Rereference
# raw_reref = raw_filt.copy().set_eeg_reference(ref, ch_type='eeg')
raw_reref = raw_filt.copy().set_eeg_reference("average", projection=False, verbose=False)

In [ ]:
raw_reref.plot(
    duration=10,
    n_channels=20,
    decim=4,
    events=events,
    event_id=event_dict,
    event_color=None,
    theme='dark',
    block=False
)

## Bad channels detection

In [ ]:
# PyPREP : find bad channels following specific criteria (correlation, deviation...)
noisy_chan = pyprep.NoisyChannels(raw)
print('Finding bad by correlation...')
noisy_chan.find_bad_by_correlation(correlation_secs=1.0,
                                   correlation_threshold=0.4,
                                   frac_bad=0.01)
print('Finding bad by deviation...')
noisy_chan.find_bad_by_deviation(deviation_threshold=5.0)
print('Finding bad by high-frequency noise...')
noisy_chan.find_bad_by_hfnoise(HF_zscore_threshold=5.0)
print('Finding bad by NaN or flat...')
noisy_chan.find_bad_by_nan_flat()
print('Finding bad by signal-to-noise ratio...')
noisy_chan.find_bad_by_SNR()

In [ ]:
bad_chan_dict = noisy_chan.get_bads(as_dict=True)
for k in bad_chan_dict:
    for idx, val in enumerate(bad_chan_dict.get(k)):
        bad_chan_dict.get(k)[idx] = str(val)
raw.info['bads'].extend(bad_chan_dict.get('bad_all'))

In [ ]:
# Replot for verification + selection if needed
raw.plot(
    duration=10,
    n_channels=20,
    decim=4,
    events=events,
    event_id=event_dict,
    event_color=None,
    theme='dark'
)

## OCULAR CORRECTION WITH ICA

ICA Parameters

In [ ]:
ica_l_freq = 1  # ICA HPF (Hz)
n_components = 0.99  # Set number of components explaining x% of variance
random_state = 42  # Set ICA seed for reproductibility
ica_method = 'fastica'
decim = 4
ica_measure = 'correlation'
ica_thresh = 0.7

Perform ICA

In [ ]:
raw_preica = raw_reref.copy().filter(ica_l_freq, h_freq=None)
epochs_preica = mne.make_fixed_length_epochs(raw_preica, duration=10)
picks = mne.pick_types(
    epochs_preica.info,
    eeg=True,
    eog=False,
    misc=False,
    exclude='bads',
)
ica = mne.preprocessing.ICA(
    n_components=n_components,
    random_state=random_state,
    method=ica_method,
    max_iter='auto',
)
ica.fit(epochs_preica, picks=picks, decim=decim, reject_by_annotation=False)

Use EOG channels for selecting components reflecting ocular artifacts (blinks + saccades)

Plot properties of components

In [ ]:
ica_scores_fig = ica.plot_scores(eog_scores)
ica.plot_sources(raw_preica, show_scrollbars=False)
ica_fig = ica.plot_components()
plt.show(block=True)

ocu_dict = {}
for ocu_components in ica.exclude:  # avoid to reprocess ICA for final report
    ocu_dict.update(
        {ocu_components: ica.plot_properties(raw_preica, picks=ocu_components, show=False)})


Apply ICA to signal

In [ ]:
raw_postica = raw_reref.copy()
ica.apply(raw_postica)

Interpolation (if bad channels detected)

In [ ]:
raw_postica.interpolate_bads(reset_bads=True) if raw_postica.info['bads'] else None

## Balistocardiogram

## Map behavior events and EEG events (checkerboard sessions)

In [ ]:
events = mne.events_from_annotations(raw)[0]
idx_trial_start = np.where(events[:,2]==2)[0]

# read behavior data
# with open("DATA\sub-001_task-CBOFF_run-1_beh.tsv") as fd:
    
#     rd = csv.reader(fd, delimiter="\t")
#     next(rd)
#     row_idx = 0
#     for row in rd:
#         if int(row[9]) != row_idx:
#             print(row_idx)
#             tasktype = row[3]
#             stimulus = row[10]
#             RT = float(row[12])
#             if tasktype == stimulus:
#                 events[idx_trial_start[row_idx],2] = 10
#             else:
#                 events[idx_trial_start[row_idx],2] = 11
#             if RT > 0:
#                 events = np.vstack([events, [events[idx_trial_start[row_idx],0]+RT*raw.info['sfreq']/1000,0,12]])
#             row_idx = row_idx + 1
# # Events
event_dict = {'stimulus_go': 10,
              'stimulus_no_go': 11,
              'response': 12}

# events_fig = mne.viz.plot_events(
#     events,
#     event_id=event_dict,
#     sfreq=raw.info['sfreq'],
#     first_samp=raw.first_samp,
#     on_missing='ignore'
# )

events_fig = mne.viz.plot_events(
    events,
    sfreq=raw.info['sfreq'],
    first_samp=raw.first_samp,
    on_missing='ignore'
)
# raw.plot(
#     duration=10,
#     n_channels=20,
#     decim=4,
#     events=events,
#     event_id=event_dict,
#     event_color=None,
#     theme='dark'
    
# )
# check number of trials

## Spectral analysis and sanity checks

In [ ]:
# Epoch = add a dimension to the data


hpf = 1 # (Hz)
lpf = 40 # (Hz)
notch = None # (Hz)
# Filters
raw.filter(hpf, lpf)
raw.notch_filter(50)

raw.pick(['Oz','O1','O2','POz'])

# psd = raw.compute_psd(
# method="welch",
# fmin=1,
# fmax=30,
# tmin=60, 
# tmax=73,
# n_fft=8192
# )

# psd = raw.compute_psd(
# method="multitaper",
# fmin=5,
# fmax=25,
# bandwidth=0.1,
# )
# psd.plot()





epochs = mne.Epochs(
raw,
events,
event_id=1,
tmin=0,
tmax=15, # or whatever the stimulus duration is
baseline=None,
preload=True,
)

psd = epochs.compute_psd(
method="multitaper",
fmin=1,
fmax=30,bandwidth=0.1,
)

psd.plot()
 
33333333333333333333333333333333333333333333

raw.drop_channels(["ECG"])

spectrum = raw.compute_psd(method='welch', fmin=0, 
fmax=100, 
tmin=165, 
tmax=180, 
)
spec_fig, axs = plt.subplots(ncols=2, figsize=(16, 7), tight_layout=True, clear=True)
axs[0] = spectrum.plot(axes=axs[0],dB=True)
axs[1] = spectrum.plot(xscale='log', axes=axs[1])
plt.show()





# # Spectral analysis
param_psd = {'method': 'welch', 'fmin': 1., 'fmax': 100.}
bands = {'Delta (1-4 Hz)': (1, 4),
'Theta (4-8 Hz)': (4, 8),
'Alpha (8-12 Hz)': (8, 12),
'Beta (12-30 Hz)': (12, 30),
'Gamma (30-45 Hz)': (30, 45)}

# Spectrum analysis: plot broad-range spectra (lin+log scales) + topomaps
# (Welch's method for computation efficiency)
spectrum = raw.compute_psd(
**param_psd,
n_fft=round(raw.info['sfreq']) * 1 # length of Welch window (pts)
)
spec_fig, axs = plt.subplots(ncols=2, figsize=(16, 7), tight_layout=True, clear=True)
axs[0] = spectrum.plot(axes=axs[0],dB=True)
axs[1] = spectrum.plot(xscale='log', axes=axs[1])
spectrum.plot_topomap(bands=bands, cmap='Spectral_r', size=10.0)
plt.show(block=True)


## Map behavior events and EEG events (SART sessions)

In [ ]:
events = mne.events_from_annotations(raw)[0]
idx_trial_start = np.where(events[:,2]==2)[0]

# read behavior data
# with open("DATA\sub-001_task-CBOFF_run-1_beh.tsv") as fd:
    
#     rd = csv.reader(fd, delimiter="\t")
#     next(rd)
#     row_idx = 0
#     for row in rd:
#         if int(row[9]) != row_idx:
#             print(row_idx)
#             tasktype = row[3]
#             stimulus = row[10]
#             RT = float(row[12])
#             if tasktype == stimulus:
#                 events[idx_trial_start[row_idx],2] = 10
#             else:
#                 events[idx_trial_start[row_idx],2] = 11
#             if RT > 0:
#                 events = np.vstack([events, [events[idx_trial_start[row_idx],0]+RT*raw.info['sfreq']/1000,0,12]])
#             row_idx = row_idx + 1
# # Events
event_dict = {'stimulus_go': 10,
              'stimulus_no_go': 11,
              'response': 12}

# events_fig = mne.viz.plot_events(
#     events,
#     event_id=event_dict,
#     sfreq=raw.info['sfreq'],
#     first_samp=raw.first_samp,
#     on_missing='ignore'
# )

events_fig = mne.viz.plot_events(
    events,
    sfreq=raw.info['sfreq'],
    first_samp=raw.first_samp,
    on_missing='ignore'
)
# raw.plot(
#     duration=10,
#     n_channels=20,
#     decim=4,
#     events=events,
#     event_id=event_dict,
#     event_color=None,
#     theme='dark'
    
# )
# check number of trials

# 2: Handling large datasets with DataLad

Visualize the raw EEG signal

# 1: EEG data preprocessing